# ENARES 2024 CRS04 — Stage 03

## Notebook 07 — Consecuencias físicas asociadas al maltrato

Traduce la sección **3.5.4** de la sintaxis SPSS y crea:

- `CONS_ALGUNA`
- `CONS_NUM_CONSECUENCIAS`
- `CONS_ATENCION_SALUD`

Preserva `NULL` fuera de los denominadores definidos por la sintaxis.

In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes

In [2]:
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path("/content/drive/MyDrive/ENARES_2024_PROJECT")
LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT_DRIVE / "02SQL"

for directory in [LOG_DIR, SQL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

print("Target table:", A)
print("RUN_UTC:", RUN_UTC)

Target table: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents
RUN_UTC: 2026-07-17T04:56:42.747081+00:00


In [4]:
required_inputs = [
    "C3P243_1", "C3P243_2", "C3P243_3",
    "C3P243_4", "C3P243_5", "C3P243_6",
    "C3P243_T1", "C3P243_T2", "C3P243_T3",
    "C3P243_T4", "C3P243_T5", "C3P243_T6",
    "VF_HOGAR", "VF_ESCUELA", "PV_VF_hogar_escuela",
]

table_before = client.get_table(A)
existing_columns = {field.name for field in table_before.schema}
missing_inputs = sorted(set(required_inputs) - existing_columns)

if missing_inputs:
    raise RuntimeError(
        "No se puede ejecutar la sección 3.5.4. Faltan variables: "
        + ", ".join(missing_inputs)
    )

if table_before.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La tabla tiene {table_before.num_rows:,} filas; "
        f"se esperaban {EXPECTED_ROWS:,}."
    )

print("Prerequisitos aprobados.")
print("Filas:", table_before.num_rows)

Prerequisitos aprobados.
Filas: 18807


In [5]:
source_variables = [
    "C3P243_1", "C3P243_2", "C3P243_3",
    "C3P243_4", "C3P243_5", "C3P243_6",
    "C3P243_T1", "C3P243_T2", "C3P243_T3",
    "C3P243_T4", "C3P243_T5", "C3P243_T6",
]

domain_parts = []

for variable in source_variables:
    domain_parts.append(f"""
    SELECT
      '{variable}' AS variable,
      CAST(`{variable}` AS STRING) AS value,
      COUNT(*) AS n
    FROM `{A}`
    GROUP BY `{variable}`
    """)

source_domain_sql = "\nUNION ALL\n".join(domain_parts)

source_domain = (
    client.query(source_domain_sql, location=LOCATION)
    .result()
    .to_dataframe()
)

source_domain.to_csv(
    LOG_DIR / "stage3_35_4_source_domain_check.csv",
    index=False,
)

display(source_domain.sort_values(["variable", "value"], na_position="first"))

,variable,value,n
0,C3P243_1,None,8224
1,C3P243_1,1,3254
2,C3P243_1,2,7329
3,C3P243_2,None,8224
5,C3P243_2,1,1259
4,C3P243_2,2,9324
6,C3P243_3,None,8224
8,C3P243_3,1,112
7,C3P243_3,2,10471
9,C3P243_4,None,8224


In [6]:
outputs = [
    "CONS_ALGUNA",
    "CONS_NUM_CONSECUENCIAS",
    "CONS_ATENCION_SALUD",
]

columns_to_replace = [
    column for column in outputs
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(f"`{column}`" for column in columns_to_replace)
        + ")"
    )
else:
    source_select = "*"

sql_consecuencias = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH base AS (
  SELECT
    {source_select},

    CASE
      WHEN
        C3P243_1 IS NOT NULL
        AND C3P243_2 IS NOT NULL
        AND C3P243_3 IS NOT NULL
        AND C3P243_4 IS NOT NULL
        AND C3P243_5 IS NOT NULL
        AND C3P243_6 IS NOT NULL
      THEN 1
      ELSE 0
    END AS _cons_all_six_valid,

    (
      CASE WHEN C3P243_1 = 1 THEN 1 ELSE 0 END
      + CASE WHEN C3P243_2 = 1 THEN 1 ELSE 0 END
      + CASE WHEN C3P243_3 = 1 THEN 1 ELSE 0 END
      + CASE WHEN C3P243_4 = 1 THEN 1 ELSE 0 END
      + CASE WHEN C3P243_5 = 1 THEN 1 ELSE 0 END
      + CASE WHEN C3P243_6 = 1 THEN 1 ELSE 0 END
    ) AS _cons_yes_count

  FROM `{A}`
),

aggregated AS (
  SELECT
    * EXCEPT(_cons_all_six_valid, _cons_yes_count),

    CASE
      WHEN _cons_all_six_valid = 0 THEN NULL
      WHEN _cons_yes_count >= 1 THEN 1
      ELSE 0
    END AS CONS_ALGUNA,

    CASE
      WHEN _cons_all_six_valid = 0 THEN NULL
      ELSE _cons_yes_count
    END AS CONS_NUM_CONSECUENCIAS

  FROM base
)

SELECT
  *,

  CASE
    WHEN CONS_ALGUNA != 1 OR CONS_ALGUNA IS NULL THEN NULL
    WHEN
      C3P243_T1 = 1
      OR C3P243_T2 = 1
      OR C3P243_T3 = 1
      OR C3P243_T4 = 1
      OR C3P243_T5 = 1
      OR C3P243_T6 = 1
    THEN 1
    ELSE 0
  END AS CONS_ATENCION_SALUD

FROM aggregated
"""

sql_path = SQL_DIR / "stage3_35_4_consecuencias.sql"
sql_path.write_text(sql_consecuencias, encoding="utf-8")

client.query(sql_consecuencias, location=LOCATION).result()

print("Variables creadas correctamente.")
print("SQL guardado en:", sql_path)

Variables creadas correctamente.
SQL guardado en: /content/drive/MyDrive/ENARES_2024_PROJECT/02SQL/stage3_35_4_consecuencias.sql


In [7]:
table_after = client.get_table(A)
columns_after = {field.name for field in table_after.schema}
missing_outputs = sorted(set(outputs) - columns_after)

if missing_outputs:
    raise RuntimeError(
        "La consulta terminó, pero faltan columnas: "
        + ", ".join(missing_outputs)
    )

if table_after.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"La transformación dejó {table_after.num_rows:,} filas; "
        f"se esperaban {EXPECTED_ROWS:,}."
    )

print("Columnas confirmadas:", ", ".join(outputs))
print("Filas preservadas:", table_after.num_rows)

Columnas confirmadas: CONS_ALGUNA, CONS_NUM_CONSECUENCIAS, CONS_ATENCION_SALUD
Filas preservadas: 18807


In [8]:
validation = (
    client.query(
        f"""
        SELECT
          COUNT(*) AS total_rows,

          COUNTIF(
            CONS_ALGUNA IS NOT NULL
            AND CONS_ALGUNA NOT IN (0, 1)
          ) AS invalid_cons_alguna,

          COUNTIF(
            CONS_NUM_CONSECUENCIAS IS NOT NULL
            AND (
              CONS_NUM_CONSECUENCIAS < 0
              OR CONS_NUM_CONSECUENCIAS > 6
            )
          ) AS invalid_cons_num,

          COUNTIF(
            CONS_ATENCION_SALUD IS NOT NULL
            AND CONS_ATENCION_SALUD NOT IN (0, 1)
          ) AS invalid_cons_atencion,

          COUNTIF(
            CONS_ATENCION_SALUD IS NOT NULL
            AND CONS_ALGUNA != 1
          ) AS atencion_outside_denominator,

          COUNTIF(
            CONS_ALGUNA = 0
            AND CONS_NUM_CONSECUENCIAS != 0
          ) AS zero_indicator_nonzero_count,

          COUNTIF(
            CONS_ALGUNA = 1
            AND CONS_NUM_CONSECUENCIAS < 1
          ) AS positive_indicator_zero_count,

          COUNTIF(
            CONS_ALGUNA IS NULL
            AND CONS_NUM_CONSECUENCIAS IS NOT NULL
          ) AS inconsistent_null_denominator,

          COUNTIF(
            PV_VF_hogar_escuela = 1
            AND NOT (VF_HOGAR = 1 AND VF_ESCUELA = 1)
          ) AS inconsistent_pv_vf

        FROM `{A}`
        """,
        location=LOCATION,
    )
    .result()
    .to_dataframe()
)

display(validation)

validation.to_csv(
    LOG_DIR / "stage3_35_4_consecuencias_validation.csv",
    index=False,
)

r = validation.iloc[0]

checks = [
    "invalid_cons_alguna",
    "invalid_cons_num",
    "invalid_cons_atencion",
    "atencion_outside_denominator",
    "zero_indicator_nonzero_count",
    "positive_indicator_zero_count",
    "inconsistent_null_denominator",
    "inconsistent_pv_vf",
]

if int(r["total_rows"]) != EXPECTED_ROWS:
    raise RuntimeError("Conteo de filas incorrecto.")

failed = {
    check: int(r[check])
    for check in checks
    if int(r[check]) > 0
}

if failed:
    raise RuntimeError(
        "Fallaron controles de calidad: "
        + ", ".join(f"{k}={v}" for k, v in failed.items())
    )

print("QA aprobado.")

,total_rows,invalid_cons_alguna,invalid_cons_num,invalid_cons_atencion,atencion_outside_denominator,zero_indicator_nonzero_count,positive_indicator_zero_count,inconsistent_null_denominator,inconsistent_pv_vf
0,18807,0,0,0,0,0,0,0,0


QA aprobado.


In [9]:
distribution_queries = {
    "CONS_ALGUNA": f"""
        SELECT
          'CONS_ALGUNA' AS variable,
          CAST(CONS_ALGUNA AS STRING) AS value,
          COUNT(*) AS n,
          ROUND(
            100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()),
            4
          ) AS percent_total
        FROM `{A}`
        GROUP BY CONS_ALGUNA
    """,
    "CONS_NUM_CONSECUENCIAS": f"""
        SELECT
          'CONS_NUM_CONSECUENCIAS' AS variable,
          CAST(CONS_NUM_CONSECUENCIAS AS STRING) AS value,
          COUNT(*) AS n,
          ROUND(
            100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()),
            4
          ) AS percent_total
        FROM `{A}`
        GROUP BY CONS_NUM_CONSECUENCIAS
    """,
    "CONS_ATENCION_SALUD": f"""
        SELECT
          'CONS_ATENCION_SALUD' AS variable,
          CAST(CONS_ATENCION_SALUD AS STRING) AS value,
          COUNT(*) AS n,
          ROUND(
            100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()),
            4
          ) AS percent_total
        FROM `{A}`
        GROUP BY CONS_ATENCION_SALUD
    """,
}

distribution_frames = []

for query in distribution_queries.values():
    distribution_frames.append(
        client.query(query, location=LOCATION)
        .result()
        .to_dataframe()
    )

distributions = pd.concat(distribution_frames, ignore_index=True)

distributions.to_csv(
    LOG_DIR / "stage3_35_4_consecuencias_distributions.csv",
    index=False,
)

display(distributions)

,variable,value,n,percent_total
0,CONS_ALGUNA,1,3935,20.9231
1,CONS_ALGUNA,None,8224,43.7284
2,CONS_ALGUNA,0,6648,35.3485
3,CONS_NUM_CONSECUENCIAS,2,889,4.7270
4,CONS_NUM_CONSECUENCIAS,None,8224,43.7284
5,CONS_NUM_CONSECUENCIAS,1,2495,13.2663
6,CONS_NUM_CONSECUENCIAS,0,6648,35.3485
7,CONS_NUM_CONSECUENCIAS,3,450,2.3927
8,CONS_NUM_CONSECUENCIAS,5,17,0.0904
9,CONS_NUM_CONSECUENCIAS,4,84,0.4466


In [10]:
cross_tab = (
    client.query(
        f"""
        SELECT
          CONS_ALGUNA,
          CONS_ATENCION_SALUD,
          COUNT(*) AS n
        FROM `{A}`
        GROUP BY
          CONS_ALGUNA,
          CONS_ATENCION_SALUD
        ORDER BY
          CONS_ALGUNA,
          CONS_ATENCION_SALUD
        """,
        location=LOCATION,
    )
    .result()
    .to_dataframe()
)

cross_tab.to_csv(
    LOG_DIR / "stage3_35_4_cons_alguna_by_atencion.csv",
    index=False,
)

display(cross_tab)

,CONS_ALGUNA,CONS_ATENCION_SALUD,n
0,<NA>,<NA>,8224
1,0,<NA>,6648
2,1,0,3556
3,1,1,379


In [11]:
closure = pd.DataFrame(
    [{
        "run_utc": RUN_UTC,
        "table": A,
        "expected_rows": EXPECTED_ROWS,
        "actual_rows": table_after.num_rows,
        "variables_created": ", ".join(outputs),
        "sql_file": str(sql_path),
        "status": "PASS",
    }]
)

closure_path = LOG_DIR / "stage3_35_4_consecuencias_closure.csv"
closure.to_csv(closure_path, index=False)

display(closure)

print("Notebook 07 completed successfully.")
print("Closure log:", closure_path)

,run_utc,table,expected_rows,actual_rows,variables_created,sql_file,status
0,2026-07-17T04:56:42.747081+00:00,enares-2024-crs04.enares2024_crs04_analytical....,18807,18807,"CONS_ALGUNA, CONS_NUM_CONSECUENCIAS, CONS_ATEN...",/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,PASS


Notebook 07 completed successfully.
Closure log: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_35_4_consecuencias_closure.csv
